In [1]:
import numpy as np  
import pandas as pd
import json 
import ast

In [16]:
df = pd.read_csv('SGJobData.csv') # lower case fixed for miniconda
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048585 entries, 0 to 1048584
Data columns (total 22 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   categories                          1044597 non-null  object 
 1   employmentTypes                     1044597 non-null  object 
 2   metadata_expiryDate                 1044597 non-null  object 
 3   metadata_isPostedOnBehalf           1048585 non-null  bool   
 4   metadata_jobPostId                  1044597 non-null  object 
 5   metadata_newPostingDate             1044597 non-null  object 
 6   metadata_originalPostingDate        1044597 non-null  object 
 7   metadata_repostCount                1048585 non-null  int64  
 8   metadata_totalNumberJobApplication  1048585 non-null  int64  
 9   metadata_totalNumberOfView          1048585 non-null  int64  
 10  minimumYearsExperience              1048585 non-null  int64  
 11  numberOfVac

In [17]:
#df.iloc[0:50000].to_excel('first 50k.xlsx')

# Cleaning up steps #
1) Remove truly blank roles
3) Remove duplicates if any.

# Remove truly blank rows #

In [18]:
# Remove truly blank rows #

categories_blank = df['categories'].isna() | (df['categories'].str.strip() == '')
employmentTypes_blank = df['employmentTypes'].isna() | (df['employmentTypes'].str.strip() == '')

print("categories blank count:", categories_blank.sum())
print("employmentTypes blank count:", employmentTypes_blank.sum())
print("both blank:", (categories_blank & employmentTypes_blank).sum())
print("categories blank but employmentTypes not:", (categories_blank & ~employmentTypes_blank).sum())
print("employmentTypes blank but categories not:", (~categories_blank & employmentTypes_blank).sum())
print("same rows (masks identical):", categories_blank.equals(employmentTypes_blank))


categories blank count: 3988
employmentTypes blank count: 3988
both blank: 3988
categories blank but employmentTypes not: 0
employmentTypes blank but categories not: 0
same rows (masks identical): True


# Remove empty rows and empty columns

In [19]:
df1 = df[df['categories'].notna() & (df['categories'].str.strip() != '')].drop(columns=['occupationId', 'status_id'])
print(df1.info())

<class 'pandas.core.frame.DataFrame'>
Index: 1044597 entries, 0 to 1048584
Data columns (total 20 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   categories                          1044597 non-null  object 
 1   employmentTypes                     1044597 non-null  object 
 2   metadata_expiryDate                 1044597 non-null  object 
 3   metadata_isPostedOnBehalf           1044597 non-null  bool   
 4   metadata_jobPostId                  1044597 non-null  object 
 5   metadata_newPostingDate             1044597 non-null  object 
 6   metadata_originalPostingDate        1044597 non-null  object 
 7   metadata_repostCount                1044597 non-null  int64  
 8   metadata_totalNumberJobApplication  1044597 non-null  int64  
 9   metadata_totalNumberOfView          1044597 non-null  int64  
 10  minimumYearsExperience              1044597 non-null  int64  
 11  numberOfVacancie

# Change data type to the right ones and change title to lower case #

In [20]:
date_cols = ['metadata_expiryDate', 'metadata_newPostingDate', 'metadata_originalPostingDate']
category_cols = ['employmentTypes', 'positionLevels', 'postedCompany_name', 'salary_type', 'status_jobStatus']

df1[date_cols] = df1[date_cols].apply(pd.to_datetime)
df1[category_cols] = df1[category_cols].astype('category')

df1['metadata_jobPostId'] = df1['metadata_jobPostId'].astype('string')
df1['title'] = df1['title'].astype('string').str.lower()

print(df1.info())

<class 'pandas.core.frame.DataFrame'>
Index: 1044597 entries, 0 to 1048584
Data columns (total 20 columns):
 #   Column                              Non-Null Count    Dtype         
---  ------                              --------------    -----         
 0   categories                          1044597 non-null  object        
 1   employmentTypes                     1044597 non-null  category      
 2   metadata_expiryDate                 1044597 non-null  datetime64[ns]
 3   metadata_isPostedOnBehalf           1044597 non-null  bool          
 4   metadata_jobPostId                  1044597 non-null  string        
 5   metadata_newPostingDate             1044597 non-null  datetime64[ns]
 6   metadata_originalPostingDate        1044597 non-null  datetime64[ns]
 7   metadata_repostCount                1044597 non-null  int64         
 8   metadata_totalNumberJobApplication  1044597 non-null  int64         
 9   metadata_totalNumberOfView          1044597 non-null  int64         
 10 

# Clean up whitespace in title and postedCompany_name #

In [21]:
# Strip leading/trailing whitespace and collapse multiple internal spaces
df1['title'] = df1['title'].str.strip().str.replace(r'\s+', ' ', regex=True)

df1['postedCompany_name'] = (
    df1['postedCompany_name']
    .astype('string')
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

print(df1[['title', 'postedCompany_name']].head())

                                               title  \
0  food technologist - clementi | entry level | u...   
1  software engineer (fab support) (java, cim, up...   
2                                  senior technician   
3  senior .net developer (.net core, mvc, mvvc, s...   
4                           sales / admin cordinator   

                 postedCompany_name  
0               WORKSTONE PTE. LTD.  
1           TRUST RECRUIT PTE. LTD.  
2        PU TIEN SERVICES PTE. LTD.  
3           TRUST RECRUIT PTE. LTD.  
4  EATZ CATERING SERVICES PTE. LTD.  


# Remove "PTE. LTD." / "LTD." from postedCompany_name and standardise to uppercase #

In [22]:
df1['postedCompany_name'] = (
    df1['postedCompany_name']
    .str.upper()
    .str.replace(r'\bPTE\.\s*LTD\.', '', regex=True)
    .str.replace(r'\bLTD\.', '', regex=True)
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .astype('category')
)

print(df1['postedCompany_name'].head(20))

0                                WORKSTONE
1                            TRUST RECRUIT
2                         PU TIEN SERVICES
3                            TRUST RECRUIT
4                   EATZ CATERING SERVICES
5                   BYTECENTURE CONSULTING
6                            TRUST RECRUIT
7                                TRITON AI
8                                WORKSTONE
9                                WORKSTONE
10                               WORKSTONE
11                               SAVORNANA
12                               WORKSTONE
13                           TRUST RECRUIT
14                           SDT MOLECULAR
15    SINGAPORE TELECOMMUNICATIONS LIMITED
16    SINGAPORE TELECOMMUNICATIONS LIMITED
17    SINGAPORE TELECOMMUNICATIONS LIMITED
18                               WORKSTONE
19                               WORKSTONE
Name: postedCompany_name, dtype: category
Categories (53130, string): ["K" LINE LOGISTICS (SINGAPORE), "K" LINE PTE LTD, #1 DESIGN STUDIO, #

# Remove duplicates again after company names are cleaned #

In [23]:
# Columns to check for duplicates (all except metadata_jobPostId)
cols_to_check = [col for col in df1.columns if col != "metadata_jobPostId"]

# Find rows that are duplicated based on those columns
# keep=False marks ALL occurrences (not just the 2nd, 3rd, etc.) as duplicates
duplicate_mask = df1.duplicated(subset=cols_to_check, keep=False)

duplicate_rows = df1[duplicate_mask]

# Optional: sort so duplicate groups sit next to each other for easy comparison
duplicate_rows = duplicate_rows.sort_values(by=cols_to_check)

print(f"Found {len(duplicate_rows)} duplicate rows")
# Drop duplicates, keeping the first occurrence of each group
df2 = df1.drop_duplicates(subset=cols_to_check, keep='first')
#print(f"Remaining rows after dropping duplicates: {len(df2)}")
df2.info()

Found 19722 duplicate rows
<class 'pandas.core.frame.DataFrame'>
Index: 1031717 entries, 0 to 1048584
Data columns (total 20 columns):
 #   Column                              Non-Null Count    Dtype         
---  ------                              --------------    -----         
 0   categories                          1031717 non-null  object        
 1   employmentTypes                     1031717 non-null  category      
 2   metadata_expiryDate                 1031717 non-null  datetime64[ns]
 3   metadata_isPostedOnBehalf           1031717 non-null  bool          
 4   metadata_jobPostId                  1031717 non-null  string        
 5   metadata_newPostingDate             1031717 non-null  datetime64[ns]
 6   metadata_originalPostingDate        1031717 non-null  datetime64[ns]
 7   metadata_repostCount                1031717 non-null  int64         
 8   metadata_totalNumberJobApplication  1031717 non-null  int64         
 9   metadata_totalNumberOfView          1031717 no

# Remove rows with implausible salary values #

Drop a row if any of the following hold:
1. Monthly **minimum** salary is below \$500 (too low to be realistic).
2. **Maximum** salary is 10x (or more) the **minimum** salary (likely a data entry error / outlier).
3. Monthly **maximum** salary is above \$50,000 (unrealistically high for the dataset).

# Justifying the salary thresholds with descriptive statistics #

Before dropping anything, check that each of the three cutoffs above actually sits in the extreme tail of the empirical distribution (using only `pandas`/`numpy` — `describe()`, `np.percentile`, and the IQR rule `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`), rather than being an arbitrary round number.

In [24]:
def iqr_bounds(s):
    q1, q3 = np.percentile(s.dropna(), [25, 75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

ratio = df2['salary_maximum'] / df2['salary_minimum'].replace(0, np.nan)

# a) Monthly minimum salary below $500
print("salary_minimum summary:")
print(df2['salary_minimum'].describe())
lo_min, hi_min = iqr_bounds(df2['salary_minimum'])
print(f"IQR outlier bounds for salary_minimum: [{lo_min:.0f}, {hi_min:.0f}]")
print(f"Percentile of 500 (% of values <= 500): {(df2['salary_minimum'] <= 500).mean() * 100:.4f}")
print(f"Rows with salary_minimum < 500: {(df2['salary_minimum'] < 500).sum()} "
      f"({(df2['salary_minimum'] < 500).mean() * 100:.4f}% of rows)")
print()

# b) Maximum salary is at least 10x the minimum salary
print("salary_maximum / salary_minimum ratio summary:")
print(ratio.describe())
lo_ratio, hi_ratio = iqr_bounds(ratio)
print(f"IQR outlier bounds for the ratio: [{lo_ratio:.2f}, {hi_ratio:.2f}]")
print(f"99th percentile of the ratio: {np.percentile(ratio.dropna(), 99):.2f}")
print(f"Rows with ratio >= 10: {(ratio >= 10).sum()} ({(ratio >= 10).mean() * 100:.4f}% of rows)")
print()

# c) Monthly maximum salary above $50,000
print("salary_maximum summary:")
print(df2['salary_maximum'].describe())
lo_max, hi_max = iqr_bounds(df2['salary_maximum'])
print(f"IQR outlier bounds for salary_maximum: [{lo_max:.0f}, {hi_max:.0f}]")
print(f"Percentile of 50000 (% of values <= 50000): {(df2['salary_maximum'] <= 50000).mean() * 100:.4f}")
print(f"Rows with salary_maximum > 50000: {(df2['salary_maximum'] > 50000).sum()} "
      f"({(df2['salary_maximum'] > 50000).mean() * 100:.4f}% of rows)")

salary_minimum summary:
count    1.031717e+06
mean     3.840029e+03
std      3.185308e+03
min      1.000000e+00
25%      2.500000e+03
50%      3.000000e+03
75%      4.500000e+03
max      3.500000e+05
Name: salary_minimum, dtype: float64
IQR outlier bounds for salary_minimum: [-500, 7500]
Percentile of 500 (% of values <= 500): 1.0737
Rows with salary_minimum < 500: 9561 (0.9267% of rows)

salary_maximum / salary_minimum ratio summary:
count    1.031717e+06
mean     6.555412e+00
std      1.128644e+03
min      1.000000e+00
25%      1.222222e+00
50%      1.384615e+00
75%      1.600000e+00
max      1.000000e+06
dtype: float64
IQR outlier bounds for the ratio: [0.66, 2.17]
99th percentile of the ratio: 2.56
Rows with ratio >= 10: 1785 (0.1730% of rows)

salary_maximum summary:
count    1.031717e+06
mean     5.756677e+03
std      5.059053e+04
min      1.000000e+00
25%      3.300000e+03
50%      4.500000e+03
75%      6.500000e+03
max      2.533000e+07
Name: salary_maximum, dtype: float64
IQR 

In [25]:
# a) Monthly minimum salary below $500
salary_too_low = df2['salary_minimum'] < 500

# b) Maximum salary is at least 10x the minimum salary
salary_ratio_too_high = df2['salary_maximum'] >= 10 * df2['salary_minimum']

# c) Monthly maximum salary above $50,000
salary_too_high = df2['salary_maximum'] > 50000

# A row is dropped if ANY of the above conditions are true
invalid_salary_mask = salary_too_low | salary_ratio_too_high | salary_too_high

print(f"Rows with minimum salary < 500: {salary_too_low.sum()}")
print(f"Rows with maximum salary >= 10x minimum salary: {salary_ratio_too_high.sum()}")
print(f"Rows with maximum salary > 50000: {salary_too_high.sum()}")
print(f"Total rows removed (union of all conditions): {invalid_salary_mask.sum()}")

df3 = df2[~invalid_salary_mask]

df3.info()

Rows with minimum salary < 500: 9561
Rows with maximum salary >= 10x minimum salary: 1785
Rows with maximum salary > 50000: 475
Total rows removed (union of all conditions): 10308
<class 'pandas.core.frame.DataFrame'>
Index: 1021409 entries, 0 to 1048574
Data columns (total 20 columns):
 #   Column                              Non-Null Count    Dtype         
---  ------                              --------------    -----         
 0   categories                          1021409 non-null  object        
 1   employmentTypes                     1021409 non-null  category      
 2   metadata_expiryDate                 1021409 non-null  datetime64[ns]
 3   metadata_isPostedOnBehalf           1021409 non-null  bool          
 4   metadata_jobPostId                  1021409 non-null  string        
 5   metadata_newPostingDate             1021409 non-null  datetime64[ns]
 6   metadata_originalPostingDate        1021409 non-null  datetime64[ns]
 7   metadata_repostCount                10214

# Detect anomalous values in minimumYearsExperience #

Detection only — no rows are dropped here yet.

In [26]:
print(df3['minimumYearsExperience'].describe())
print()

# Full distribution of values, sorted by the value itself
print(df3['minimumYearsExperience'].value_counts().sort_index())
print()

# Candidate anomalies: negative years (impossible)
negative_years = df3['minimumYearsExperience'] < 0
print(f"Rows with negative minimumYearsExperience: {negative_years.sum()}")

# Candidate anomalies: implausibly high years (e.g. >= 30 years required for a job posting)
excessive_years = df3['minimumYearsExperience'] >= 30
print(f"Rows with minimumYearsExperience >= 30: {excessive_years.sum()}")

count    1.021409e+06
mean     2.805459e+00
std      2.534225e+00
min      0.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      4.000000e+00
max      8.800000e+01
Name: minimumYearsExperience, dtype: float64

minimumYearsExperience
0     109140
1     255700
2     217575
3     168026
4      34746
5     140896
6      20082
7      11475
8      27227
9       1162
10     26298
11       205
12      2592
13       203
14       156
15      4679
16        68
17        33
18       116
19        17
20       745
21         5
22        21
23         5
24         2
25       110
26         4
27         4
28         3
29         3
30        79
31         4
32         2
33         3
35         6
38         1
40         2
47         1
50         3
55         2
56         1
59         1
61         1
62         1
76         1
87         2
88         1
Name: count, dtype: int64

Rows with negative minimumYearsExperience: 0
Rows with minimumYearsExperience >= 30: 111


# Remove rows with anomalous minimumYearsExperience #

Drop rows where `minimumYearsExperience >= 30`, based on the empirical distribution of this column in `df3` (i.e. the percentile is computed directly from the observed values, not from a fitted/theoretical distribution such as a normal distribution).

Computed via pandas boolean comparisons (`<=`/`<` combined with `.mean()`), a value of 30 sits at roughly the **99.99th percentile** (99.997 using the "weak" definition — the percentage of values <= 30 — versus 99.989 using the "strict" definition — the percentage of values < 30). Either way, `minimumYearsExperience >= 30` is comfortably in the extreme top ~0.01% tail of the data, consistent with the earlier finding that these are almost certainly data entry errors.

In [27]:
# Percentile of the value 30 within the empirical distribution of minimumYearsExperience
# ("weak" = % of values <= 30, "strict" = % of values < 30 — same definitions scipy.stats.percentileofscore uses)
pct_weak = (df3['minimumYearsExperience'] <= 30).mean() * 100
pct_strict = (df3['minimumYearsExperience'] < 30).mean() * 100
print(f"Percentile of 30 (% of values <= 30): {pct_weak:.4f}")
print(f"Percentile of 30 (% of values < 30): {pct_strict:.4f}")

# Drop rows with implausible minimumYearsExperience
excessive_years_mask = df3['minimumYearsExperience'] >= 30
print(f"\nRows removed (minimumYearsExperience >= 30): {excessive_years_mask.sum()}")

df4 = df3[~excessive_years_mask]

df4.info()

Percentile of 30 (% of values <= 30): 99.9969
Percentile of 30 (% of values < 30): 99.9891

Rows removed (minimumYearsExperience >= 30): 111
<class 'pandas.core.frame.DataFrame'>
Index: 1021298 entries, 0 to 1048574
Data columns (total 20 columns):
 #   Column                              Non-Null Count    Dtype         
---  ------                              --------------    -----         
 0   categories                          1021298 non-null  object        
 1   employmentTypes                     1021298 non-null  category      
 2   metadata_expiryDate                 1021298 non-null  datetime64[ns]
 3   metadata_isPostedOnBehalf           1021298 non-null  bool          
 4   metadata_jobPostId                  1021298 non-null  string        
 5   metadata_newPostingDate             1021298 non-null  datetime64[ns]
 6   metadata_originalPostingDate        1021298 non-null  datetime64[ns]
 7   metadata_repostCount                1021298 non-null  int64         
 8   metada

# Final duplicate/inconsistency check on repost metadata #

Drop a row if either of the following hold — both are inconsistencies between `metadata_repostCount` and the posting dates:
1. `metadata_repostCount` is positive (job was reposted), but `metadata_newPostingDate` and `metadata_originalPostingDate` are the same date (no new posting date was actually recorded).
2. `metadata_repostCount` is 0 (job was never reposted), but `metadata_newPostingDate` differs from `metadata_originalPostingDate` (a new posting date exists despite no repost).

In [28]:
# Compare dates only (ignore any time component)
same_date = df4['metadata_newPostingDate'].dt.date == df4['metadata_originalPostingDate'].dt.date

# 1) Repost claimed but dates identical
repost_but_same_date = (df4['metadata_repostCount'] > 0) & same_date

# 2) No repost claimed but dates differ
no_repost_but_diff_date = (df4['metadata_repostCount'] == 0) & ~same_date

inconsistent_repost_mask = repost_but_same_date | no_repost_but_diff_date

print(f"Rows with repostCount > 0 but same newPostingDate/originalPostingDate: {repost_but_same_date.sum()}")
print(f"Rows with repostCount == 0 but different newPostingDate/originalPostingDate: {no_repost_but_diff_date.sum()}")
print(f"Total rows removed (union of both conditions): {inconsistent_repost_mask.sum()}")

df5 = df4[~inconsistent_repost_mask]

df5.info()

Rows with repostCount > 0 but same newPostingDate/originalPostingDate: 81
Rows with repostCount == 0 but different newPostingDate/originalPostingDate: 0
Total rows removed (union of both conditions): 81
<class 'pandas.core.frame.DataFrame'>
Index: 1021217 entries, 0 to 1048574
Data columns (total 20 columns):
 #   Column                              Non-Null Count    Dtype         
---  ------                              --------------    -----         
 0   categories                          1021217 non-null  object        
 1   employmentTypes                     1021217 non-null  category      
 2   metadata_expiryDate                 1021217 non-null  datetime64[ns]
 3   metadata_isPostedOnBehalf           1021217 non-null  bool          
 4   metadata_jobPostId                  1021217 non-null  string        
 5   metadata_newPostingDate             1021217 non-null  datetime64[ns]
 6   metadata_originalPostingDate        1021217 non-null  datetime64[ns]
 7   metadata_repostCou

In [29]:
df5.to_csv('SGJobData_cleaned.csv', index=False)